# Automatic Music Transcription
* Convert audio mp3 files to ascii gutar tabs

# TODO
* Consider using an embedding layer at the input, as it can help in learning a more meaningful representation of the tokens


In [ ]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Preprocessing

In [3]:
# Load in the mp3 file, get the sample rate, and get the data
# from the file
#

import numpy as np
import matplotlib.pyplot as plt
import librosa

# Load in the mp3 file
y, sr = librosa.load('dataset/john-mayer/3_Why_Georgia.mp3')

In [4]:
# Get the data from the file
data = librosa.feature.mfcc(y=y, sr=sr)

In [17]:
def plotMFCC(data):
    plt.figure(figsize=(10, 4))
    librosa.display.specshow(data, x_axis='time')
    plt.colorbar()
    plt.title('MFCC')
    plt.tight_layout()
    plt.show()

# plotMFCC(data)

In [7]:
# install pyguitarpro
# !pip install pyguitarpro
# !pip install --upgrade attrs


In [9]:
import guitarpro

# Load in the gp5 file
song = guitarpro.parse('dataset/john-mayer/3_why-georgia.gp5')

In [12]:
song.tracks

In [25]:
token_note_format = 'instrument:note:string:fret'
token_rest_format = 'instrument:rest'

song.tempo, song.key

(98, <KeySignature.CMajor: (0, 0)>)

In [26]:
def tokenize_note_effect(note_effect):
    effect_tokens = []
    
    if note_effect.accentuatedNote:
        effect_tokens.append('nfx:accentuated')
        
    if note_effect.bend:
        effect_tokens.append(f'nfx:bend:{note_effect.bend.type}')
        
    if note_effect.ghostNote:
        effect_tokens.append('nfx:ghost')
        
    if note_effect.grace:
        effect_tokens.append(f'nfx:grace:{note_effect.grace.fret}:{note_effect.grace.duration}:{note_effect.grace.transition}')
        
    if note_effect.hammer:
        effect_tokens.append('nfx:hammer')
        
    if note_effect.harmonic:
        # Assuming harmonic.type exists or similar attributes to represent harmonic type
        effect_tokens.append(f'nfx:harmonic:{type(note_effect.harmonic).__name__}')
        
    if note_effect.heavyAccentuatedNote:
        effect_tokens.append('nfx:heavy_accentuated')
        
    if note_effect.leftHandFinger != 'open':
        effect_tokens.append(f'nfx:left_finger:{note_effect.leftHandFinger}')
        
    if note_effect.letRing:
        effect_tokens.append('nfx:let_ring')
        
    if note_effect.palmMute:
        effect_tokens.append('nfx:palm_mute')
        
    if note_effect.rightHandFinger != 'open':
        effect_tokens.append(f'nfx:right_finger:{note_effect.rightHandFinger}')
        
    if note_effect.slides:
        for slide in note_effect.slides:
            effect_tokens.append(f'nfx:slide:{slide}')
            
    if note_effect.staccato:
        effect_tokens.append('nfx:staccato')
        
    if note_effect.tremoloPicking:
        effect_tokens.append(f'nfx:tremolo:{note_effect.tremoloPicking.duration.value}')
        
    if note_effect.trill:
        effect_tokens.append(f'nfx:trill:{note_effect.trill.fret}:{note_effect.trill.duration.value}')
        
    if note_effect.vibrato:
        effect_tokens.append('nfx:vibrato')
        
    return effect_tokens


In [43]:
def tokenize_song(song):
    tokens = []
    
    # Start tokens
    tokens.append(f'artist:{song.artist}')
    tokens.append(f'downtune:{song.tracks[0].strings[0].value}')  # Assuming first track, first string for downtune info
    tokens.append(f'tempo:{song.tempo}')
    tokens.append('start')
    tokens.append('new_measure')

    for track in song.tracks:
        for measure in track.measures:
            for voice in measure.voices:
                for beat in voice.beats:
                    
                    if track.isPercussionTrack:
                        pass    # TODO Skip drums for now
                        # tokens.append(f'drums:note:{beat.notes[0].value}')  # Assuming one note per beat for drums
                    else:
                        # Note tokens for pitched instruments (guitar, bass, etc. not drums)
                        for note in beat.notes:
                            tokens.append(f'{track.name}:note:s{note.string}:f{note.value}')

                            if isinstance(note.effect, guitarpro.NoteEffect):  # This part is pseudo-code; adapt based on actual PyGuitarPro attributes
                                note_effect_token = tokenize_note_effect(note.effect)
                                tokens.append(f'nfx:{note_effect_token}')
                    
                    # Assuming 'wait' tokens are in terms of beats; adapt as needed
                    tokens.append(f'wait:{beat.duration.value}')

                    # Rest tokens
                    if beat.status == guitarpro.BeatStatus.rest:
                        tokens.append(f'{track.name}:note:rest')
                        
                    # Add any beat effects, note effects, tempo changes, etc.
                    # ... (this part depends on what effects you want to include)
    tokens.append('end')
    return tokens

# Load the song using PyGuitarPro (pseudo-code)
# song = guitarpro.parse('path/to/your/file.gp5')

# Tokenize the song
# tokenize_song(song)

# Print the tokens (or save them, etc.)
# print(tokens)


In [35]:
import copy

# set acoustic_song to song but copy the value so we don't change the original
acoustic_song = copy.deepcopy(song)

In [39]:
acoustic_song.tracks = [acoustic_song.tracks[0]]

In [40]:
song.tracks, acoustic_song.tracks

([<guitarpro.models.Track at 0x7fbdc0aa69d0>,
 [<guitarpro.models.Track at 0x7fbdd0c280d0>])

In [41]:
print(len(song.tracks), len(acoustic_song.tracks))

6 1


In [56]:
dir(acoustic_song)

['__annotations__',
 '__attrs_attrs__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slotnames__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_currentRepeatGroup',
 'addMeasureHeader',
 'album',
 'artist',
 'clipboard',
 'copyright',
 'hideTempo',
 'instructions',
 'key',
 'lyrics',
 'masterEffect',
 'measureHeaders',
 'music',
 'newMeasure',
 'notice',
 'pageSetup',
 'subtitle',
 'tab',
 'tempo',
 'tempoName',
 'title',
 'tracks',
 'version',
 'versionTuple',
 'words']

In [61]:
acoustic_song.title, acoustic_song.artist, acoustic_song.tempo, acoustic_song.key

('Why Georgia', 'John Mayer', 98, <KeySignature.CMajor: (0, 0)>)

In [76]:
def count_acoustic_notes():
    acoustic_notes = []
    max_num_notes = 50
    counter = 0

    for track in acoustic_song.tracks:
        for measure in track.measures:
            for voice in measure.voices:
                for beat in voice.beats:
                    for note in beat.notes:
                        acoustic_notes.append(note)

                        counter += 1
                        if counter > max_num_notes:
                            return acoustic_notes

acoustic_notes = count_acoustic_notes()

In [79]:
len(acoustic_notes)
acoustic_notes

[Note(value=0, velocity=47, string=4, effect=<guitarpro.models.NoteEffect object at 0x7fbda02d93d0>, durationPercent=1.0, swapAccidentals=False, type=<NoteType.dead: 3>),
 Note(value=0, velocity=47, string=5, effect=<guitarpro.models.NoteEffect object at 0x7fbda02d94f0>, durationPercent=1.0, swapAccidentals=False, type=<NoteType.dead: 3>),
 Note(value=3, velocity=79, string=6, effect=<guitarpro.models.NoteEffect object at 0x7fbda02d95b0>, durationPercent=1.0, swapAccidentals=False, type=<NoteType.normal: 1>),
 Note(value=3, velocity=79, string=2, effect=<guitarpro.models.NoteEffect object at 0x7fbda02d99a0>, durationPercent=1.0, swapAccidentals=False, type=<NoteType.normal: 1>),
 Note(value=0, velocity=79, string=3, effect=<guitarpro.models.NoteEffect object at 0x7fbda02d9a60>, durationPercent=1.0, swapAccidentals=False, type=<NoteType.normal: 1>),
 Note(value=2, velocity=79, string=3, effect=<guitarpro.models.NoteEffect object at 0x7fbda02d9df0>, durationPercent=1.0, swapAccidentals=F

In [44]:
tokens = tokenize_song(acoustic_song)

In [45]:
tokens

['artist:John Mayer',
 'downtune:64',
 'tempo:98',
 'start',
 'new_measure',
 'wait:4',
 'Guitar 1:note:rest',
 'Guitar 1:note:s4:f0',
 "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
 'Guitar 1:note:s5:f0',
 "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
 'Guitar 1:note:s6:f3',
 "nfx:['nfx:left_finger:Fingering.open', 'nfx:let_ring', 'nfx:right_finger:Fingering.open']",
 'wait:8',
 'Guitar 1:note:s2:f3',
 "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
 'Guitar 1:note:s3:f0',
 "nfx:['nfx:hammer', 'nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
 'wait:16',
 'Guitar 1:note:s3:f2',
 "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
 'wait:16',
 'Guitar 1:note:s5:f0',
 "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
 'Guitar 1:note:s6:f0',
 "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
 'wait:1

In [46]:
print(tokenize_note_effect(notes[0].effect))

notes[0]

['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']


Note(value=0, velocity=47, string=4, effect=<guitarpro.models.NoteEffect object at 0x7fbdb2d987c0>, durationPercent=1.0, swapAccidentals=False, type=<NoteType.dead: 3>)

In [48]:
len(tokens)

9090

In [50]:
# Initialize an empty dictionary to store the mapping from tokens to integers
token_to_int = {}
int_to_token = {}
next_int = 0  # Start assigning from 0

# Function to dynamically update the dictionary
def update_token_mapping(token, token_to_int, int_to_token, next_int):
    if token not in token_to_int:
        token_to_int[token] = next_int
        int_to_token[next_int] = token
        next_int += 1
    return next_int

# Update the dictionary with the tokens from the file
for token in tokens:
    next_int = update_token_mapping(token, token_to_int, int_to_token, next_int)

# Show some statistics and samples to verify
num_unique_tokens = len(token_to_int)
sample_items_token_to_int = list(token_to_int.items())[:15]
sample_items_int_to_token = list(int_to_token.items())[:15]

num_unique_tokens, sample_items_token_to_int, sample_items_int_to_token


(51,
 [('artist:John Mayer', 0),
  ('downtune:64', 1),
  ('tempo:98', 2),
  ('start', 3),
  ('new_measure', 4),
  ('wait:4', 5),
  ('Guitar 1:note:rest', 6),
  ('Guitar 1:note:s4:f0', 7),
  ("nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']",
   8),
  ('Guitar 1:note:s5:f0', 9),
  ('Guitar 1:note:s6:f3', 10),
  ("nfx:['nfx:left_finger:Fingering.open', 'nfx:let_ring', 'nfx:right_finger:Fingering.open']",
   11),
  ('wait:8', 12),
  ('Guitar 1:note:s2:f3', 13),
  ('Guitar 1:note:s3:f0', 14)],
 [(0, 'artist:John Mayer'),
  (1, 'downtune:64'),
  (2, 'tempo:98'),
  (3, 'start'),
  (4, 'new_measure'),
  (5, 'wait:4'),
  (6, 'Guitar 1:note:rest'),
  (7, 'Guitar 1:note:s4:f0'),
  (8,
   "nfx:['nfx:left_finger:Fingering.open', 'nfx:right_finger:Fingering.open']"),
  (9, 'Guitar 1:note:s5:f0'),
  (10, 'Guitar 1:note:s6:f3'),
  (11,
   "nfx:['nfx:left_finger:Fingering.open', 'nfx:let_ring', 'nfx:right_finger:Fingering.open']"),
  (12, 'wait:8'),
  (13, 'Guitar 1:note:s2:f

In [51]:
# Convert the tokens to their corresponding integer values using the mapping
integer_representation = [token_to_int[token] for token in tokens]

# Show some sample integer representation of the tokens
integer_representation[:10], integer_representation[-10:]


([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [14, 8, 7, 8, 9, 8, 10, 8, 49, 50])

In [52]:
integer_representation

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 8,
 10,
 11,
 12,
 13,
 8,
 14,
 15,
 16,
 17,
 8,
 16,
 9,
 8,
 18,
 8,
 16,
 13,
 11,
 14,
 11,
 16,
 9,
 11,
 12,
 19,
 11,
 20,
 11,
 12,
 14,
 11,
 16,
 7,
 8,
 9,
 8,
 16,
 20,
 11,
 16,
 19,
 11,
 16,
 14,
 8,
 16,
 7,
 8,
 9,
 8,
 10,
 11,
 12,
 13,
 8,
 14,
 15,
 16,
 17,
 8,
 16,
 9,
 8,
 18,
 8,
 16,
 13,
 11,
 14,
 11,
 16,
 9,
 11,
 12,
 19,
 11,
 20,
 11,
 12,
 14,
 11,
 16,
 7,
 8,
 9,
 8,
 16,
 20,
 11,
 16,
 19,
 11,
 16,
 14,
 8,
 16,
 7,
 8,
 9,
 8,
 10,
 11,
 12,
 13,
 8,
 14,
 15,
 16,
 17,
 8,
 16,
 9,
 8,
 18,
 8,
 16,
 13,
 11,
 14,
 11,
 16,
 9,
 11,
 12,
 19,
 11,
 20,
 11,
 12,
 14,
 11,
 16,
 7,
 8,
 9,
 8,
 16,
 20,
 11,
 16,
 19,
 11,
 16,
 14,
 8,
 16,
 7,
 8,
 9,
 8,
 10,
 11,
 12,
 13,
 8,
 14,
 15,
 16,
 17,
 8,
 16,
 9,
 8,
 18,
 8,
 16,
 13,
 11,
 14,
 11,
 16,
 9,
 11,
 12,
 19,
 11,
 20,
 11,
 12,
 14,
 11,
 16,
 7,
 8,
 9,
 8,
 16,
 20,
 11,
 16,
 19,
 11,
 16,
 14,
 8,
 16,
 7,
 8,
 9,
 8,
 10,
 11,
 12,
 

In [53]:
len(integer_representation)

9090

In [54]:
max(integer_representation)

50

Load the gp5 file into guitar pro so we can visualize the file and see what we are working with.  We can also use this to generate the midi file.

In [ ]:
first_track = song.tracks[0]

first_track_attributes = dir(first_track)

first_track_important_attributes = ['channel',
 'color',
 'fretCount',
 'indicateTuning',
 'is12StringedGuitarTrack',
 'isBanjoTrack',
 'isMute',
 'isPercussionTrack',
 'isSolo',
 'isVisible',
 'measures',
 'name',
 'number',
 'offset',
 'port',
 'rse',
 'settings',
 'song',
 'strings',
 'useRSE']

for attribute in first_track_important_attributes:
    print(attribute, getattr(first_track, attribute))
